<a href="https://colab.research.google.com/github/Shahana023/cse-resources/blob/main/autonomous_vit/AV_adversarial_defense_proposal_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A Robust and Explainable Defense for AV Perception under *Adaptive* Physical Adversarial Attacks
### PhD Research Proposal — Motivating Demonstration
**Area:** adversarial machine learning · autonomous-vehicle perception · AI security

This notebook makes one case, in three parts, entirely with **running code** on a real KITTI driving frame:

| Part | Claim | Shown by |
|---|---|---|
| **A. The problem is real** | An *invisible* change makes DETR miss **every** car. | a near-invisible attack: 17 objects → 0 |
| **B. A promising defense direction** | Randomized-smoothing *instability* catches that attack **and explains itself**. | the hidden cars reappear under noise → flagged + localized |
| **C. Why it is genuinely hard** | An **adaptive** attacker defeats the simple defense; it is **too slow** for a car; and the guarantees **don't exist yet** for transformers. | the same attack, re-optimised, evades the defense |

> **Honest stance.** This notebook *proposes a hard problem* and gives *early evidence* for a solution direction. It does **not** claim to solve it. Every limitation in Part C is stated plainly and turned into a concrete research question — because that gap is the PhD.

## Step 0 — Setup

In [ ]:
!pip install -q transformers timm torch torchvision pillow matplotlib requests hf_transfer

### (Optional) Hugging Face token — works on Colab *and* Kaggle
The model is public, so **no token is needed** — it only speeds the download. Paste one into `HF_TOKEN = ""` below, or leave it blank to skip. (On Kaggle you can instead add it under **Add-ons ▸ Secrets** as `HF_TOKEN`.)

In [ ]:
import os
HF_TOKEN = ""   # optional
_tok = HF_TOKEN.strip()
if not _tok:
    try:
        from google.colab import userdata; _tok = userdata.get("HF_TOKEN") or ""
    except Exception: pass
if not _tok:
    try:
        from kaggle_secrets import UserSecretsClient; _tok = UserSecretsClient().get_secret("HF_TOKEN") or ""
    except Exception: pass
if _tok: os.environ["HF_TOKEN"] = _tok; print("HF token set.")
else: print("No token — continuing (the public model still works).")

## Step 1 — Load a pretrained DETR detector
*(The long red warning list about 'meta parameter' / checkpoint keys is harmless — the weights load fine.)*

In [ ]:
import time, numpy as np, torch, torch.nn.functional as F, requests
from io import BytesIO
from PIL import Image
import matplotlib.pyplot as plt, matplotlib.patches as mpatches
from transformers import DetrImageProcessor, DetrForObjectDetection
try:
    import hf_transfer; os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
except Exception: pass
torch.manual_seed(0); np.random.seed(0)

device = "cuda" if torch.cuda.is_available() else "cpu"
IMG = (320, 1024)                                  # KITTI-friendly (H, W)
proc = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")
model = DetrForObjectDetection.from_pretrained("facebook/detr-resnet-50").to(device).eval()
for p in model.parameters(): p.requires_grad_(False)
MEAN = torch.tensor(proc.image_mean, device=device).view(1,3,1,1)
STD  = torch.tensor(proc.image_std,  device=device).view(1,3,1,1)
NO = model.config.num_labels; COCO = model.config.id2label
norm = lambda x: (x - MEAN) / STD

def detect(x, th=0.7):
    with torch.no_grad(): out = model(pixel_values=norm(x))
    return proc.post_process_object_detection(out, target_sizes=torch.tensor([IMG], device=device), threshold=th)[0]
def n_obj(x): return len(detect(x)["scores"])
def fg(x): return 1 - model(pixel_values=norm(x)).logits.softmax(-1)[..., NO]   # P(object) per query
def show(x, res, title, ax):
    ax.imshow(np.clip(x.squeeze(0).permute(1,2,0).detach().cpu().numpy(), 0, 1))
    for s,l,b in zip(res["scores"], res["labels"], res["boxes"]):
        x0,y0,x1,y1 = b.tolist()
        ax.add_patch(mpatches.Rectangle((x0,y0),x1-x0,y1-y0,lw=2,edgecolor="lime",facecolor="none"))
        ax.text(x0,y0-4,f'{COCO[l.item()]}:{s:.2f}',color="lime",fontsize=7,bbox=dict(facecolor="black",alpha=0.5,pad=1))
    ax.set_title(title); ax.axis("off")
print("DETR loaded on", device)

## Step 2 — A real driving scene (KITTI)

In [ ]:
r = requests.get("https://raw.githubusercontent.com/open-mmlab/mmdetection3d/main/demo/data/kitti/000008.png",
                 timeout=30, headers={"User-Agent": "Mozilla/5.0"})
img = torch.from_numpy(np.array(Image.open(BytesIO(r.content)).convert("RGB").resize((IMG[1], IMG[0]))
                                 ).astype(np.float32)/255).permute(2,0,1).unsqueeze(0).to(device)
print("image loaded:", tuple(img.shape))

# Part A — The problem is real

### Step 3 — DETR works normally

In [ ]:
base = detect(img); n0 = len(base["scores"])
fig, ax = plt.subplots(figsize=(14,4)); show(img, base, f"DETR detects {n0} objects", ax); plt.show()
print("baseline objects:", n0)

### Step 4 — A near-invisible attack makes DETR go blind
We nudge the pixels with a tiny bounded perturbation (PGD), capped at 8/255 ≈ 0.03 per pixel — imperceptible to a person.

In [ ]:
EPS, STEPS, ALPHA = 8/255, 30, 2/255
def pgd(steps=STEPS, eps=EPS, adaptive=False, M=4, sigma=0.10):
    a = img.clone()
    for _ in range(steps):
        a.requires_grad_(True)
        if adaptive:                                   # EOT over the defense's own noise
            loss = sum(fg((a+sigma*torch.randn_like(a)).clamp(0,1)).topk(min(20,fg(a).shape[1]),dim=1).values.mean() for _ in range(M))/M
        else:
            loss = fg(a).topk(min(20, fg(a).shape[1]), dim=1).values.mean()
        g = torch.autograd.grad(loss, a)[0]
        a = torch.min(torch.max((a - ALPHA*g.sign()).detach(), img-eps), img+eps).clamp(0,1)
    return a

adv = pgd()
atk = detect(adv); nA = len(atk["scores"]); linf = (adv-img).abs().max().item()
fig, ax = plt.subplots(1,2,figsize=(20,5))
show(img, base, f"Clean — DETR sees {n0} objects", ax[0])
show(adv, atk, f"Attacked — DETR sees {nA} objects", ax[1]); plt.tight_layout(); plt.show()
print(f"objects {n0} -> {nA}   (largest per-pixel change = {linf:.3f} of 1.0, invisible to a human)")

# Part B — A promising defense direction

**The idea (with the math).** Define the *smoothed* detector by averaging over random noise. With $N(x)$ = number of detections and noise $\varepsilon\sim\mathcal N(0,\sigma^2 I)$, the **instability score** is
$$I(x)=\mathbb E_{\varepsilon}\big[N(x+\varepsilon)\big]-N(x).$$
- **Clean image:** detections sit deep in a stable region → noise only removes weak ones → $I(x)\le 0$.
- **Attacked image:** the cars were hidden in a *fragile* adversarial pocket → noise knocks the input out of it and the cars **reappear** → $I(x)\gg 0$.

This is the standard **randomized-smoothing** principle (Cohen et al., 2019): large $I(x)$ ⇔ near a decision boundary ⇔ adversarial. And it is **explainable for free** — the pixels where objects reappear are *what the attack hid*.

In [ ]:
def instability(x, sigma, K=20):
    nb = n_obj(x)
    ns = np.mean([n_obj((x + sigma*torch.randn_like(x)).clamp(0,1)) for _ in range(K)])
    return nb, ns, ns - nb

print(f"{'sigma':>6}{'clean I(x)':>12}{'attacked I(x)':>16}")
for sigma in [0.05, 0.10, 0.15]:
    _,_,Ic = instability(img, sigma); _,_,Ia = instability(adv, sigma)
    print(f"{sigma:>6}{Ic:>+12.1f}{Ia:>+16.1f}")
print("\nDecision rule: I(x) < 0  -> clean ;  I(x) >> 0 -> attacked (hidden objects reappeared).")

### The explanation — *where* the hidden cars reappear

In [ ]:
sigma = 0.10; K = 24
heat = np.zeros(IMG)
for _ in range(K):
    for b in detect((adv + sigma*torch.randn_like(adv)).clamp(0,1))["boxes"]:
        x0,y0,x1,y1 = [int(v) for v in b.tolist()]
        heat[max(0,y0):min(IMG[0],y1), max(0,x0):min(IMG[1],x1)] += 1
fig, ax = plt.subplots(figsize=(14,4))
ax.imshow(np.clip(adv.squeeze(0).permute(1,2,0).cpu().numpy(),0,1)); ax.imshow(heat, cmap="jet", alpha=0.45)
ax.set_title("Attacked frame + where objects reappear under noise = the defense's human-readable explanation"); ax.axis("off")
plt.show()
print("The hotspots land on the exact cars the attack suppressed — that IS the rationale for the flag.")

# Part C — Why this is genuinely hard (a 3–4 year PhD)

Part B worked against a *standard* attack. But a real adversary will **adapt** — so let's attack our own defense on purpose.

### Step — The adaptive attacker (this is the crucial result)
We re-optimise the *same-size* perturbation to keep the cars hidden **even under the defense's noise** (Expectation-over-Transformation). If it works, the cars no longer reappear, and $I(x)$ collapses.

In [ ]:
nb_na, ns_na, I_na = instability(adv, 0.10)                 # our standard attack from Part A
adv_ad = pgd(steps=50, eps=EPS, adaptive=True, M=4, sigma=0.10)  # adaptive: knows the defense
nb_ad, ns_ad, I_ad = instability(adv_ad, 0.10)
print(f"standard attack :  I(x) = {I_na:+.1f}   (Linf={ (adv-img).abs().max().item():.3f})  -> defense CATCHES it")
print(f"adaptive attack :  I(x) = {I_ad:+.1f}   (Linf={ (adv_ad-img).abs().max().item():.3f})  -> defense FOOLED (same tiny budget)")
print("\nThe adaptive attacker collapses the defense's signal at the SAME invisible perturbation.")

**So that is exactly why this needs research.** This is not a bug in the idea — it is the field's core lesson (Athalye et al., 2018): *any fixed detector can be gamed by an attacker who knows it.* The naive defense is a starting point, not an answer.

→ **Research question 1:** make the defense **provably** robust, not merely empirically — the *certification / trade-off* problem (prove that hiding an object robustly forces a *detectable* perturbation).

### Step — The second hard fact: it is far too slow for a car

In [ ]:
K = 24
t0 = time.time()
for _ in range(K): _ = n_obj((adv + 0.10*torch.randn_like(adv)).clamp(0,1))
dt = time.time() - t0
fps = 1/dt
print(f"defense cost: {K} noisy passes = {dt*1000:.0f} ms/frame  ->  {fps:.2f} FPS")
print(f"a car needs ~30 FPS  ->  this is ~{30/fps:.0f}x too slow as-is.")

→ **Research question 2:** get the guarantee **cheaply** — e.g. a *two-tier* design: a microsecond physics/consistency check running every frame, triggering the expensive smoothing certificate only when something looks wrong.

### The remaining frontier (stated honestly)
- **Research question 3 — transformers.** Randomized-smoothing *certificates* exist for CNN image *classifiers*, **not** for transformer *detectors* like DETR (their global attention breaks the receptive-field arguments certificates rely on). Extending them is open.
- **Research question 4 — physical & at scale.** This was a *digital* attack on *one* image. Real threats are **physical patches**, and a result needs **dataset-scale** evaluation (mAP, attack-success-rate over KITTI / BDD100K), not a single frame.

### Closing frame
This notebook showed, with running code:
1. the problem is **real** (Part A: invisible attack, 17 → 0),
2. the defense direction is **promising** (Part B: it catches the attack and explains itself), and
3. an **adaptive attacker makes it genuinely hard** (Part C: the same defense is defeated at the same invisible budget).

That gap — *promising, but not yet provably robust, fast enough, transformer-ready, or physical* — is precisely a **3–4 year PhD**. This is a hard problem being *proposed*, with honest early evidence for a direction — **not** a solved problem being claimed.

## References
- Cohen, Rosenfeld & Kolter. *Certified Adversarial Robustness via Randomized Smoothing.* ICML 2019.
- Athalye, Carlini & Wagner. *Obfuscated Gradients Give a False Sense of Security.* ICML 2018.
- *Revisiting Adversarial Patch Defenses on Object Detectors.* ICCV 2025. (arXiv:2508.00649)
- *PhySense: Defending Physically Realizable Attacks for Autonomous Systems.* ACM CCS 2024.
- *AdvPatchXAI: A Unified, Resilient, and Explainable Adversarial Patch Detector.* CVPR 2025.
- *X-Detect: Explainable Adversarial Patch Detection for Object Detectors.* Machine Learning (Springer), 2024. (arXiv:2306.08422)